# MICO Set-up 현황 조회
SQLite DB에서 Category / SubCategory / Detail 데이터를 DataFrame으로 추출

In [1]:
import sqlite3
import pandas as pd

# DB 경로 (이 노트북이 notebooks/ 폴더에 있을 경우)
DB_PATH = '../db.sqlite3'

conn = sqlite3.connect(DB_PATH)
print('DB 연결 완료')

DB 연결 완료


## 1. 각 테이블 단독 조회

In [2]:
# Category
df_category = pd.read_sql_query("""
    SELECT id, product, oper_id, oper_desc, created_at
    FROM setup_mico_category
    ORDER BY product, oper_id
""", conn)

print(f'Category: {len(df_category)}건')
df_category

Category: 3건


,id,product,oper_id,oper_desc,created_at
0,4,LC,A850740C,M1 CU CMP,2026-03-17 23:57:42.523071
1,1,LC,A908740A,SN BPSG CMP,2026-03-16 12:50:42.073715
2,6,LC,T550000A,M2 CU CMP,2026-03-19 13:28:27.128045


In [3]:
# SubCategory
df_subcategory = pd.read_sql_query("""
    SELECT id, category_id, fab, device, recipe_id, maker, created_at
    FROM setup_mico_subcategory
    ORDER BY category_id, fab, device
""", conn)

print(f'SubCategory: {len(df_subcategory)}건')
df_subcategory

SubCategory: 4건


,id,category_id,fab,device,recipe_id,maker,created_at
0,1,1,M14,E2,E2_SNBPSG_R03_AB,EBARA,2026-03-16 12:50:42.074800
1,5,1,M14,E9,E9_SNBPSG_R03_AB,EBARA,2026-03-17 01:03:08.203630
2,9,4,M14,E2,E2_M1CU_R12,AMAT,2026-03-18 04:23:03.747112
3,10,6,M14,E2,E2_M2CU_R12,AMAT,2026-03-19 13:29:14.922523


In [4]:
# Detail
df_detail = pd.read_sql_query("""
    SELECT id, subcategory_id,
           apc_para, thk_para,
           target, pre_target, pre_thk_period,
           rr_para, rr_max, rr_period, if_rr,
           offset_group,
           created_at
    FROM setup_mico_detail
    ORDER BY subcategory_id
""", conn)

print(f'Detail: {len(df_detail)}건')
df_detail

Detail: 14건


,id,subcategory_id,apc_para,thk_para,target,pre_target,pre_thk_period,rr_para,rr_max,rr_period,if_rr,offset_group,created_at
0,1,1,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y,2026-03-16 13:18:05.071038
1,2,1,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y,2026-03-16 13:20:05.919045
2,3,1,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y,2026-03-16 13:20:13.628418
3,4,5,PB_04_A8,EBARA_ITM_POST_THK1_ED2_AVG,2800,3500,10,pad,None,None,None,Y,2026-03-17 01:03:08.205684
4,5,5,PB_04_A7,EBARA_ITM_POST_THK1_ED1_AVG,2800,3500,10,pad,None,None,None,Y,2026-03-17 01:03:08.207323
5,6,5,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3400,10,pad,None,None,None,Y,2026-03-17 01:03:08.208185
6,10,9,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y,2026-03-18 04:23:03.748773
7,11,9,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y,2026-03-18 04:23:03.749884
8,12,9,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y,2026-03-18 04:23:03.750792
9,13,9,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3400,10,pad,None,None,None,Y,2026-03-19 04:22:25.273639


## 2. 전체 계층 JOIN (Category → SubCategory → Detail)

In [5]:
df_full = pd.read_sql_query("""
    SELECT
        c.product,
        c.oper_id,
        c.oper_desc,
        s.fab,
        s.device,
        s.recipe_id,
        s.maker,
        d.apc_para,
        d.thk_para,
        d.target,
        d.pre_target,
        d.pre_thk_period,
        d.rr_para,
        d.rr_max,
        d.rr_period,
        d.if_rr,
        d.offset_group
    FROM setup_mico_detail      AS d
    JOIN setup_mico_subcategory AS s ON d.subcategory_id = s.id
    JOIN setup_mico_category    AS c ON s.category_id   = c.id
    ORDER BY c.product, c.oper_id, s.fab, s.device, s.recipe_id
""", conn)

print(f'전체 Set-up: {len(df_full)}건')
df_full

전체 Set-up: 14건


,product,oper_id,oper_desc,fab,device,recipe_id,maker,apc_para,thk_para,target,pre_target,pre_thk_period,rr_para,rr_max,rr_period,if_rr,offset_group
0,LC,A850740C,M1 CU CMP,M14,E2,E2_M1CU_R12,AMAT,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
1,LC,A850740C,M1 CU CMP,M14,E2,E2_M1CU_R12,AMAT,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
2,LC,A850740C,M1 CU CMP,M14,E2,E2_M1CU_R12,AMAT,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
3,LC,A850740C,M1 CU CMP,M14,E2,E2_M1CU_R12,AMAT,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3400,10,pad,None,None,None,Y
4,LC,A908740A,SN BPSG CMP,M14,E2,E2_SNBPSG_R03_AB,EBARA,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
5,LC,A908740A,SN BPSG CMP,M14,E2,E2_SNBPSG_R03_AB,EBARA,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
6,LC,A908740A,SN BPSG CMP,M14,E2,E2_SNBPSG_R03_AB,EBARA,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
7,LC,A908740A,SN BPSG CMP,M14,E9,E9_SNBPSG_R03_AB,EBARA,PB_04_A8,EBARA_ITM_POST_THK1_ED2_AVG,2800,3500,10,pad,None,None,None,Y
8,LC,A908740A,SN BPSG CMP,M14,E9,E9_SNBPSG_R03_AB,EBARA,PB_04_A7,EBARA_ITM_POST_THK1_ED1_AVG,2800,3500,10,pad,None,None,None,Y
9,LC,A908740A,SN BPSG CMP,M14,E9,E9_SNBPSG_R03_AB,EBARA,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3400,10,pad,None,None,None,Y


## 3. 조건 필터링 예시

In [ ]:
# product로 필터
TARGET_PRODUCT = 'LC'   # ← 원하는 product 입력

df_filtered = df_full[df_full['product'] == TARGET_PRODUCT].reset_index(drop=True)
print(f'{TARGET_PRODUCT} → {len(df_filtered)}건')
df_filtered

In [ ]:
# product + oper_id 로 필터
TARGET_PRODUCT = 'LC'
TARGET_OPER_ID = 'T550000A'   # ← 원하는 oper_id 입력

df_filtered2 = df_full[
    (df_full['product'] == TARGET_PRODUCT) &
    (df_full['oper_id'] == TARGET_OPER_ID)
].reset_index(drop=True)

print(f'{TARGET_PRODUCT} / {TARGET_OPER_ID} → {len(df_filtered2)}건')
df_filtered2

## 4. pandas merge 방식 (SQL 대신 Python으로 JOIN)

In [6]:
df_merge = (
    df_detail
    .merge(df_subcategory.rename(columns={'id': 'subcategory_id'}),
           on='subcategory_id', suffixes=('', '_sub'))
    .merge(df_category.rename(columns={'id': 'category_id'}),
           on='category_id', suffixes=('', '_cat'))
)

# 필요한 컬럼만 정리
cols = ['product', 'oper_id', 'oper_desc',
        'fab', 'device', 'recipe_id', 'maker',
        'apc_para', 'thk_para', 'target', 'pre_target', 'pre_thk_period',
        'rr_para', 'rr_max', 'rr_period', 'if_rr', 'offset_group']
df_merge = df_merge[cols].sort_values(['product', 'oper_id', 'fab', 'device']).reset_index(drop=True)

print(f'전체 Set-up: {len(df_merge)}건')
df_merge

전체 Set-up: 14건


,product,oper_id,oper_desc,fab,device,recipe_id,maker,apc_para,thk_para,target,pre_target,pre_thk_period,rr_para,rr_max,rr_period,if_rr,offset_group
0,LC,A850740C,M1 CU CMP,M14,E2,E2_M1CU_R12,AMAT,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
1,LC,A850740C,M1 CU CMP,M14,E2,E2_M1CU_R12,AMAT,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
2,LC,A850740C,M1 CU CMP,M14,E2,E2_M1CU_R12,AMAT,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
3,LC,A850740C,M1 CU CMP,M14,E2,E2_M1CU_R12,AMAT,PB_04_TIME,AMAT_ITM_POST_THK1_AVG,2800,3400,10,pad,None,None,None,Y
4,LC,A908740A,SN BPSG CMP,M14,E2,E2_SNBPSG_R03_AB,EBARA,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
5,LC,A908740A,SN BPSG CMP,M14,E2,E2_SNBPSG_R03_AB,EBARA,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
6,LC,A908740A,SN BPSG CMP,M14,E2,E2_SNBPSG_R03_AB,EBARA,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3500,10,pad,None,None,None,Y
7,LC,A908740A,SN BPSG CMP,M14,E9,E9_SNBPSG_R03_AB,EBARA,PB_04_A8,EBARA_ITM_POST_THK1_ED2_AVG,2800,3500,10,pad,None,None,None,Y
8,LC,A908740A,SN BPSG CMP,M14,E9,E9_SNBPSG_R03_AB,EBARA,PB_04_A7,EBARA_ITM_POST_THK1_ED1_AVG,2800,3500,10,pad,None,None,None,Y
9,LC,A908740A,SN BPSG CMP,M14,E9,E9_SNBPSG_R03_AB,EBARA,PB_04_TIME,EBARA_ITM_POST_THK1_AVG,2800,3400,10,pad,None,None,None,Y


## 5. 요약 통계

In [7]:
print('=== Category별 SubCategory / Detail 수 ===')
summary = (
    df_full
    .groupby(['product', 'oper_id', 'oper_desc'])
    .agg(
        subcategory_count=('recipe_id', 'nunique'),
        detail_count=('apc_para', 'count')
    )
    .reset_index()
)
summary

=== Category별 SubCategory / Detail 수 ===


,product,oper_id,oper_desc,subcategory_count,detail_count
0,LC,A850740C,M1 CU CMP,1,4
1,LC,A908740A,SN BPSG CMP,2,6
2,LC,T550000A,M2 CU CMP,1,4


In [8]:
# 연결 종료
conn.close()
print('DB 연결 종료')

DB 연결 종료
